# Объединение CSV Equaring (2023–2025)

Сливает 6 полугодовых файлов в один `merged_2023_2025.csv`.

Базовые проверки:
- одинаковый набор колонок во всех файлах,
- итоговое число строк после `concat` равно сумме строк по файлам.

In [ ]:
import os
import pandas as pd

DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"
FILES = [
    "jan-jun2023.csv",
    "jul-dec2023.csv",
    "jan-jun2024.csv",
    "jul-dec2024.csv",
    "jan-jun2025.csv",
    "jul-dec2025.csv",
]
OUT_FILE = f"{DATA_DIR}/merged_2023_2025.csv"

print("DATA_DIR:", DATA_DIR)
print("OUT_FILE:", OUT_FILE)
print("Файлов:", len(FILES))

In [ ]:
# Диагностика: как выглядят данные после пропуска 75 строк.
# Для каждого файла показываем:
#  - 76-ю строку (заголовок),
#  - первые 3 строки данных (77-79),
#  - количество ';' в заголовке и в первых строках данных,
#  - последнюю строку файла (хвост).
PREVIEW_DATA_LINES = 3
SEP = ";"
SKIP_ROWS = 75

for name in FILES:
    path = f"{DATA_DIR}/{name}"
    print("=" * 80)
    print(f"Файл: {name}")
    print("=" * 80)

    with open(path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    total_lines = len(lines)
    print(f"Всего строк в файле: {total_lines:,}")
    print(f"Строк после skiprows={SKIP_ROWS}: {max(total_lines - SKIP_ROWS, 0):,}")

    if total_lines <= SKIP_ROWS:
        print("В файле меньше строк, чем SKIP_ROWS — нечего показывать.")
        continue

    header_line = lines[SKIP_ROWS].rstrip("\n")
    header_seps = header_line.count(SEP)
    print(f"\nЗаголовок (стр. {SKIP_ROWS + 1}), ';' = {header_seps}, колонок = {header_seps + 1}:")
    print(header_line[:500] + ("..." if len(header_line) > 500 else ""))

    print("\nПервые строки данных:")
    for i in range(SKIP_ROWS + 1, min(SKIP_ROWS + 1 + PREVIEW_DATA_LINES, total_lines)):
        raw = lines[i].rstrip("\n")
        seps = raw.count(SEP)
        diff = seps - header_seps
        flag = "" if diff == 0 else f"  [!] лишних ';' = {diff}"
        print(f"  стр. {i + 1}: ';' = {seps}{flag}")
        print(f"    {raw[:400]}" + ("..." if len(raw) > 400 else ""))

    last_raw = lines[-1].rstrip("\n")
    last_seps = last_raw.count(SEP)
    print(f"\nПоследняя строка (стр. {total_lines}), ';' = {last_seps}:")
    print(f"  {last_raw[:400]}" + ("..." if len(last_raw) > 400 else ""))
    print()

In [ ]:
# В первых 75 строках каждого файла лежит скрипт выгрузки,
# реальные данные начинаются с 76-й строки (она содержит заголовки колонок).
# Разделитель в файлах — точка с запятой.
# Загружаем БЕЗ потерь строк: если в строке лишние ';', склеиваем их обратно в COMMENT_TEXT.
SKIP_ROWS = 75
SEP = ";"


def read_semicolon_keep_all(path, skip_rows=75, sep=";"):
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    if len(lines) <= skip_rows:
        raise ValueError(f"Недостаточно строк после skip_rows={skip_rows}: {path}")

    header = lines[skip_rows].rstrip("\n")
    columns = header.split(sep)
    ncols = len(columns)

    try:
        comment_idx = columns.index("COMMENT_TEXT")
    except ValueError:
        comment_idx = None

    records = []
    fixed_more = 0
    fixed_less = 0

    for raw_line in lines[skip_rows + 1 :]:
        raw = raw_line.rstrip("\n")
        parts = raw.split(sep)

        if len(parts) == ncols:
            records.append(parts)
            continue

        if len(parts) > ncols and comment_idx is not None:
            # Лишние разделители считаем частью COMMENT_TEXT
            extra = len(parts) - ncols
            merged_comment = sep.join(parts[comment_idx : comment_idx + extra + 1])
            repaired = parts[:comment_idx] + [merged_comment] + parts[comment_idx + extra + 1 :]
            if len(repaired) == ncols:
                records.append(repaired)
                fixed_more += 1
                continue

        if len(parts) < ncols:
            # Если полей меньше, дополняем пустыми справа.
            records.append(parts + [""] * (ncols - len(parts)))
            fixed_less += 1
            continue

        # Редкий fallback: берем первые ncols, остаток приклеиваем в последний столбец.
        tail = sep.join(parts[ncols - 1 :])
        repaired = parts[: ncols - 1] + [tail]
        records.append(repaired)
        fixed_more += 1

    df = pd.DataFrame(records, columns=columns)
    return df, ncols, fixed_more, fixed_less


dfs = []
stats = []

for name in FILES:
    path = f"{DATA_DIR}/{name}"
    if not os.path.exists(path):
        raise FileNotFoundError(f"Файл не найден: {path}")

    df, ncols, fixed_more, fixed_less = read_semicolon_keep_all(path, skip_rows=SKIP_ROWS, sep=SEP)
    dfs.append(df)
    stats.append({
        "файл": name,
        "строк": len(df),
        "колонок": df.shape[1],
        "исправлено_лишних_разделителей": fixed_more,
        "дополнено_коротких_строк": fixed_less,
    })
    print(
        f"{name}: {len(df):,} строк, {df.shape[1]} колонок, "
        f"исправлено(лишние ';')={fixed_more}, дополнено(короткие)={fixed_less}"
    )

stats_df = pd.DataFrame(stats)
print("\nИтого по файлам:")
display(stats_df)

In [ ]:
base_cols = set(dfs[0].columns)
mismatch = {}
for name, df in zip(FILES, dfs):
    cols = set(df.columns)
    only_here = cols - base_cols
    only_in_base = base_cols - cols
    if only_here or only_in_base:
        mismatch[name] = {
            "лишние": sorted(only_here),
            "отсутствуют": sorted(only_in_base),
        }

if mismatch:
    print("Расхождения по колонкам относительно первого файла:")
    for name, diff in mismatch.items():
        print(f"\n{name}:")
        print(f"  лишние:      {diff['лишние']}")
        print(f"  отсутствуют: {diff['отсутствуют']}")
    raise AssertionError("Набор колонок различается между файлами — слияние остановлено.")

print("Все файлы имеют одинаковый набор колонок:", len(base_cols), "шт.")

expected_rows = sum(len(df) for df in dfs)
print(f"Ожидаемое число строк после concat: {expected_rows:,}")

In [ ]:
merged = pd.concat(dfs, ignore_index=True)

assert len(merged) == expected_rows, (
    f"После concat строк: {len(merged):,}, ожидалось: {expected_rows:,}"
)

merged.to_csv(OUT_FILE, index=False, encoding="utf-8")

print("Готово.")
print(f"Файл:    {OUT_FILE}")
print(f"Строк:   {len(merged):,}")
print(f"Колонок: {merged.shape[1]}")